In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sentencepiece as spm
import itertools
import json
import random
import tqdm
from torch.optim import AdamW, lr_scheduler
from torch.amp import autocast, GradScaler
import math
import numpy as np
import torch.multiprocessing as mp
mp.set_start_method("fork", force=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

sentinels = [f"▁<sen_{i}>" for i in range(100)]
VOCAB_SIZE = 8192
N_WEB = 300_000
N_JSON = 200_000

13.0
running on cuda


web = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train",
    streaming=True,
)

with open("trained_tokenizer/corpus.txt", "w", encoding="utf-8") as f:

    print("Writing web text...")
    for ex in tqdm(itertools.islice(web, N_WEB)):
        text = ex.get("text", "").strip()
        if text:
            f.write(text.replace("\n", " ") + "\n")

KEYS   = ["name","type","value","id","url","path","query","status","code",
          "message","data","result","error","count","limit","offset","text",
          "title","body","tags","meta","config","params","args","kwargs"]
TYPES  = ["string","integer","number","boolean","array","object","null"]
WORDS  = ["hello","world","example","test","foo","bar","baz","user","admin",
          "true","false","null","yes","no","ok","success","failure","pending"]

def rand_val(depth=0):
    if depth > 3:
        return random.choice([
            random.randint(0, 9999),
            random.choice(WORDS),
            True, False, None,
        ])
    t = random.choice(["str","int","bool","null","list","dict"])
    if t == "str":   return random.choice(WORDS)
    if t == "int":   return random.randint(0, 9999)
    if t == "bool":  return random.choice([True, False])
    if t == "null":  return None
    if t == "list":  return [rand_val(depth+1) for _ in range(random.randint(1,4))]
    if t == "dict":  return rand_obj(depth+1)

def rand_obj(depth=0):
    n = random.randint(1, 6)
    return {random.choice(KEYS): rand_val(depth) for _ in range(n)}

print("Generating JSON corpus...")
with open("trained_tokenizer/corpus_json.txt", "w") as f:
    for _ in tqdm(range(200_000)):
        f.write(json.dumps(rand_obj()) + "\n")

spm.SentencePieceTrainer.train(
    input=["trained_tokenizer/corpus.txt", "trained_tokenizer/corpus_json.txt"],
    model_prefix="trained_tokenizer/tokenizer",
    vocab_size=8192,
    model_type="bpe",
    character_coverage=0.9995,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    pad_piece="[PAD]",
    unk_piece="[UNK]",
    bos_piece="[BOS]",
    eos_piece="[EOS]",
    user_defined_symbols = sentinels,
    input_sentence_size=100000,
    shuffle_input_sentence=True,
    minloglevel=-1
)

In [2]:
sp = spm.SentencePieceProcessor()
sp.load(f"trained_tokenizer/tokenizer.model")

tests = [
    '{"name": "get_weather", "arguments": {"location": "SF"}}',
    'what even <sen_0> is this',
    '{"status": "success", "data": [{"id": 1, "value": "foo"}]}',
    '{"error": null, "count": 42, "result": true}',
]
print("\nSanity check:")
for s in tests:
    toks = sp.encode(s, out_type=int)
    print(f"  {len(toks):2d} tok | {toks}")

print(type(toks))


Sanity check:
  23 tok | [153, 378, 110, 112, 898, 8183, 648, 991, 190, 112, 247, 8110, 671, 110, 153, 8107, 301, 202, 110, 112, 8126, 8154, 2302]
   5 tok | [592, 805, 4, 180, 300]
  20 tok | [153, 425, 110, 112, 738, 190, 112, 415, 110, 561, 167, 110, 2923, 112, 409, 110, 112, 747, 504, 539]
  15 tok | [153, 390, 110, 169, 8115, 112, 383, 110, 366, 607, 112, 412, 110, 209, 8130]
<class 'list'>


In [3]:
class RotaryPositionalEmbeddings(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, offset=0):
        B, N, H, D = x.shape
        t = torch.arange(offset, offset + N, device=x.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos()[None, :, None, :]   # 1, N, 1, D
        sin = emb.sin()[None, :, None, :]
        return x * cos + self._rotate_half(x) * sin

    def _rotate_half(self, x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_k_heads):
        super().__init__()

        assert d_model % num_k_heads == 0
        assert d_model % num_q_heads == 0
        assert num_q_heads % num_k_heads == 0

        self.num_q = num_q_heads
        self.num_k = num_k_heads
        self.head_dim = int(d_model // num_q_heads)
        self.scale = self.head_dim ** 0.5

        self.q_to_k = int(num_q_heads // num_k_heads)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_kv = nn.Linear(d_model, self.num_k * self.head_dim * 2, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryPositionalEmbeddings(self.head_dim)

    def forward(self, x, attn_mask = None): # Attn mask is B, N
        B, N, _ = x.shape

        Q = self.W_q(x) # B, N, d_model
        KV = self.W_kv(x) # B, N, d_model

        Q = Q.view(B, N, self.num_q, self.head_dim) # B N H D
        KV = KV.view(B, N, 2, self.num_k, self.head_dim).permute(2, 0, 1, 3, 4)
        K, V = KV[0], KV[1] # B N H/qk D

        Q = self.rope(Q).permute(0, 2, 1, 3) # B H N D
        K = self.rope(K).permute(0, 2, 1, 3) # B H/qk N D
        V = V.permute(0, 2, 1, 3) # B H/qk N D

        K_full = torch.repeat_interleave(K, repeats=self.q_to_k, dim = 1) # B H N D
        V_full = torch.repeat_interleave(V, repeats=self.q_to_k, dim = 1)

        score = (Q @ K_full.transpose(2,3)) / self.scale

        if attn_mask is not None:
            score = score.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))

        weights = F.softmax(score, dim = -1)
        weights = torch.nan_to_num(weights, nan=0.0)
        out = weights @ V_full # B H N D
        out = out.permute(0, 2, 1, 3).contiguous() # B N H D
        out = out.view(B, N, -1)

        return self.W_o(out), K, V

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_q_heads, num_k_heads, dropout = 0.2):
        super().__init__()
        self.attn = GroupedQueryAttention(d_model, num_q_heads, num_k_heads)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model)
        )

        self.ln1 = nn.RMSNorm(d_model)
        self.ln2 = nn.RMSNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask = None):
        attended, K, V = self.attn(self.ln1(x), attn_mask)
        x = x + self.dropout(attended)
        h = self.ln2(x)
        x = x + self.dropout(self.ff(h))

        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_layers=8, num_q_heads=8, num_k_heads=4, dropout = 0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformers = nn.ModuleList([
            EncoderLayer(self.d_model, num_q_heads = num_q_heads, num_k_heads = num_k_heads, dropout = dropout) 
            for _ in range(num_layers)
        ])

    def forward(self, x, attn_mask = None):
        out = self.embed(x)
        for t in self.transformers:
            out = t(out, attn_mask)

        return out

In [4]:
class MaskedGroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_k_heads):
        super().__init__()

        assert d_model % num_k_heads == 0
        assert d_model % num_q_heads == 0
        assert num_q_heads % num_k_heads == 0

        self.num_q = num_q_heads
        self.num_k = num_k_heads
        self.head_dim = int(d_model // num_q_heads)
        self.scale = self.head_dim ** 0.5

        self.q_to_k = int(num_q_heads // num_k_heads)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_kv = nn.Linear(d_model, self.num_k * self.head_dim * 2, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryPositionalEmbeddings(self.head_dim)

    def forward(self, x, attn_mask = None, kv_cache = None): # Attn mask is B, N
        # print("Causal GQA ", type(kv_cache), kv_cache)
        B, N, _ = x.shape

        Q = self.W_q(x) # B, N, d_model
        KV = self.W_kv(x) # B, N, d_model

        Q = Q.view(B, N, self.num_q, self.head_dim) # B N H D
        KV = KV.view(B, N, 2, self.num_k, self.head_dim).permute(2, 0, 1, 3, 4)
        K, V = KV[0], KV[1] # B N H/qk D

        offset = kv_cache["K"].shape[2] if kv_cache is not None else 0

        Q = self.rope(Q, offset).permute(0, 2, 1, 3) # B H N D
        K = self.rope(K, offset).permute(0, 2, 1, 3) # B H/qk N D
        V = V.permute(0, 2, 1, 3) # B H/qk N D

        if kv_cache is not None:
            K = torch.cat([kv_cache["K"], K], dim = 2) # appends N to past token count
            V = torch.cat([kv_cache["V"], V], dim = 2)

        K_full = torch.repeat_interleave(K, repeats=self.q_to_k, dim = 1) # B H N D
        V_full = torch.repeat_interleave(V, repeats=self.q_to_k, dim = 1)

        score = (Q @ K_full.transpose(2,3)) / self.scale
        mask = torch.triu(torch.ones(N, K.shape[2], device=x.device), diagonal=K.shape[2] - N + 1).bool()
        score = score.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        if attn_mask is not None:
            score = score.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))        

        weights = F.softmax(score, dim = -1)
        weights = torch.nan_to_num(weights, nan=0.0)
        out = weights @ V_full # B H N D
        out = out.permute(0, 2, 1, 3).contiguous() # B N H D
        out = out.view(B, N, -1)

        return self.W_o(out), {"K" : K, "V" : V}
    
class GroupedQueryCrossAttention(nn.Module):
    def __init__(self, d_model, num_q_heads, num_k_heads):
        super().__init__()

        assert d_model % num_k_heads == 0
        assert d_model % num_q_heads == 0
        assert num_q_heads % num_k_heads == 0

        self.num_q = num_q_heads
        self.num_k = num_k_heads
        self.head_dim = int(d_model // num_q_heads)
        self.scale = self.head_dim ** 0.5

        self.q_to_k = int(num_q_heads // num_k_heads)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_kv = nn.Linear(d_model, self.num_k * self.head_dim * 2, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_dec, x_enc, attn_mask = None, kv_cache = None): # Attn mask is B, S
        B, T, _ = x_dec.shape # T is dec seq len

        Q = self.W_q(x_dec) # B, T, d_model
        Q = Q.view(B, T, self.num_q, self.head_dim).permute(0, 2, 1, 3) # B H T D

        if kv_cache is None:
            B, S, _ = x_enc.shape # S is enc seq len
            KV = self.W_kv(x_enc) # B, S, d_model
            KV = KV.view(B, S, 2, self.num_k, self.head_dim).permute(2, 0, 3, 1, 4)
            K, V = KV[0], KV[1] # B H/qk S D
        else:
            K = kv_cache["K"]
            V = kv_cache["V"]

        K_full = torch.repeat_interleave(K, repeats=self.q_to_k, dim = 1) # B H S D
        V_full = torch.repeat_interleave(V, repeats=self.q_to_k, dim = 1)

        score = (Q @ K_full.transpose(2,3)) / self.scale # B H T S

        if attn_mask is not None:
            score = score.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))

        weights = F.softmax(score, dim = -1)
        weights = torch.nan_to_num(weights, nan=0.0)
        out = weights @ V_full # B H T D
        out = out.permute(0, 2, 1, 3).reshape(B, T, -1) # B T H D, B T D

        return self.W_o(out), {"K" : K, "V" : V}
    
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_q_heads, num_k_heads, dropout = 0.2):
        super().__init__()
        self.d_model = d_model
        self.s_attn = MaskedGroupedQueryAttention(d_model, num_q_heads, num_k_heads)
        self.x_attn = GroupedQueryCrossAttention(d_model, num_q_heads, num_k_heads)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model)
        )

        self.ln1 = nn.RMSNorm(d_model)
        self.ln2 = nn.RMSNorm(d_model)
        self.ln3 = nn.RMSNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x_dec, x_enc, attn_mask = None, x_attn_mask = None, self_kv_cache = None, cross_kv_cache = None):
        # print("decoder layer ", self_kv_cache)
        attended, self_kv = self.s_attn(self.ln1(x_dec), attn_mask, self_kv_cache)
        x_dec = x_dec + self.dropout(attended)

        x_attended, cross_kv = self.x_attn(self.ln2(x_dec), x_enc, x_attn_mask, cross_kv_cache)
        x_dec = x_dec + self.dropout(x_attended)

        h = self.ln3(x_dec)
        x_dec = x_dec + self.dropout(self.ff(h))

        return x_dec, self_kv, cross_kv

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_layers=8, num_q_heads=8, num_k_heads=4, dropout = 0.2):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformers = nn.ModuleList([
            DecoderLayer(self.d_model, num_q_heads = num_q_heads, num_k_heads = num_k_heads, dropout = dropout) 
            for _ in range(num_layers)
        ])

        self.unembed = nn.Linear(d_model, vocab_size, bias = False)
        self.unembed.weight = self.embed.weight

        self.num_layers = num_layers


    def forward(self, x_self, x_enc, caches = None, dec_attn_mask = None, enc_attn_mask = None): # cross attend to final encoder output
        new_cache = []
        out = self.embed(x_self)
        for i, t in enumerate(self.transformers):

            layer_cache = {} if caches is None else caches[i]
            # print("decoder", layer_cache.get("self"))

            out, kv_dec, kv_cross = t(out, x_enc, dec_attn_mask, enc_attn_mask, layer_cache.get("self"), layer_cache.get("cross"))
            new_cache.append({"self" : kv_dec, "cross" : kv_cross})

        return self.unembed(out), new_cache

In [5]:
class EncoderQueryDecoderGeneratorModel(nn.Module):
    def __init__(self, vocab_size, d_model, enc_layers, dec_layers, num_q_heads=8, num_k_heads=4, dropout = 0.2):
        super().__init__()

        self.encoder = Encoder(
            vocab_size=vocab_size, 
            d_model=d_model,
            num_layers=enc_layers,
            num_q_heads=num_q_heads,
            num_k_heads=num_k_heads,
            dropout=dropout
        )

        self.decoder = Decoder(
            vocab_size=vocab_size,
            d_model=d_model,
            num_layers=dec_layers,
            num_q_heads=num_q_heads,
            num_k_heads=num_k_heads,
            dropout=dropout
        )

        self.decoder.embed.weight = self.encoder.embed.weight
        self.decoder.unembed.weight = self.encoder.embed.weight

    def forward(self, x_enc, x_dec, enc_mask = None, dec_mask = None):
        enc_out = self.encoder(x_enc, attn_mask = enc_mask)
        dec_out, _ = self.decoder(x_dec, enc_out, dec_attn_mask = dec_mask, enc_attn_mask = enc_mask)
        return dec_out
    
    @torch.no_grad()
    def generate(self, x_enc, enc_mask = None, max_new_tokens = 128, end_token_id = 3, start_token_id = 2, temp = 1, top_k = 16):
        B, _ = x_enc.shape
        encoded = self.encoder(x_enc, enc_mask)

        caches = None
        tokens = []
        cur_token = torch.full((B, 1), start_token_id, device=device, dtype=torch.long)
        for i in range(max_new_tokens):
            logits, caches = self.decoder(cur_token, encoded, caches = caches)
            logits = logits[:, -1, :]

            logits = logits / temp
            topk_vals = torch.topk(logits, top_k).values[:, -1, None]
            logits = logits.masked_fill(logits < topk_vals, float("-inf"))
            token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)

            tokens.append(token)
            if (token == end_token_id).all():
                break

        return torch.cat(tokens, dim = 1)

In [7]:
SEQ_LEN = 512
TOTAL_TOKENS = 1_000_000_000
BATCH_SIZE = 32
ACCUM_STEPS = 8
LR = 3e-4
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LOG_STEPS = 5
SAVE_STEPS = 100
CKPT_DIR = "./checkpoints"

whereswaldo = EncoderQueryDecoderGeneratorModel(
    vocab_size=sp.vocab_size(),
    d_model=512,
    enc_layers=8,
    dec_layers=8,
    num_k_heads=4,
    num_q_heads=8,
).to(device)

print(f"params: {sum(p.numel() for p in whereswaldo.parameters())}")

optimizer = AdamW(whereswaldo.parameters(), lr=LR, weight_decay=0.01)
total_steps = TOTAL_TOKENS // (BATCH_SIZE * SEQ_LEN)
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps: return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler()

nn.init.normal_(whereswaldo.encoder.embed.weight, std=0.02)

print(total_steps)

params: 56684544
61035


In [8]:
class PretrainDS(Dataset):
    def __init__(self, tokenizer : spm.SentencePieceProcessor, max_len = 512, min_len=20, ds_name = "allenai/c4", split = "train", buffer_size = 25000, total_batches = total_steps):
        super().__init__()

        self.tokenizer = tokenizer
        self.pad_id = self.tokenizer.pad_id()
        self.bos_id = self.tokenizer.bos_id()
        self.eos_id = self.tokenizer.eos_id()
        self.sent_ids = [self.tokenizer.piece_to_id(f"▁<sen_{i}>") for i in range(100)]
        self.max_len = max_len
        self.min_len = min_len
        self.buffer_size = buffer_size
        self.ds_name = ds_name
        self.split = split
        self.total_batches = total_batches

    def prepare_seq(self, tokenized, p = 1/5, mask_pct = 0.2, min_start = 2):
        num = int(mask_pct * len(tokenized))
        spans = []
        while num > 0:
            new_span = min(np.random.geometric(p), num)
            num -= new_span
            spans.append(new_span)
        
        spans = np.array(spans)
        starts_raw = np.sort(np.random.choice(len(tokenized) - spans.sum() - min_start, size=spans.shape[0], replace=False))
        starts = starts_raw + np.concatenate([[min_start], spans[:-1].cumsum()])

        masked_parts = []

        for sen in range(spans.shape[0]-1, -1, -1):
            masked_parts.append(
                [self.sent_ids[sen]] + tokenized[starts[sen]:starts[sen] + spans[sen]]
            )
            tokenized = tokenized[:starts[sen]] + [self.sent_ids[sen]] + tokenized[starts[sen] + spans[sen]:]

        masked_parts = sum(masked_parts[::-1], [])

        return tokenized, [self.tokenizer.bos_id()] + masked_parts, masked_parts + [self.tokenizer.eos_id()]
        
    def __getitem__(self, index):
        while True:
            try:
                text = next(self.iterator)["text"]
                tokens = self.tokenizer.encode(text, out_type = int)
                sample = tokens[:self.max_len]
                if len(sample) > self.min_len:
                    in_seq, out_seq, out_lab = self.prepare_seq(sample)
                    return {
                        "x_enc" : in_seq,
                        "x_dec" : out_seq,
                        "dec_lab" : out_lab,
                    }
            except:
                self.iterator = iter(self.dataset)

    def __len__(self):
        return self.total_batches

In [9]:
def collate_fn(batch, PAD_ID = sp.pad_id()):
    enc = [torch.tensor(b["x_enc"]) for b in batch]
    dec = [torch.tensor(b["x_dec"]) for b in batch]
    out = [torch.tensor(b["dec_lab"]) for b in batch]

    enc_padded = pad_sequence(enc, batch_first=True, padding_value=PAD_ID)
    dec_padded = pad_sequence(dec, batch_first=True, padding_value=PAD_ID)
    out_padded = pad_sequence(out, batch_first=True, padding_value=PAD_ID)

    return ({
        "x_enc" : enc_padded,
        "x_dec" : dec_padded,
        "enc_mask" : (enc_padded != PAD_ID),
        "dec_mask" : (dec_padded != PAD_ID)
    }, out_padded)

def worker_init_fn(worker_id):
    worker_info = torch.utils.data.get_worker_info()
    ds = worker_info.dataset
    ds.dataset = load_dataset(ds.ds_name, "en", split=ds.split, streaming=True).shuffle(buffer_size=1000)
    ds.iterator = iter(ds.dataset)

ds = PretrainDS(sp, max_len=SEQ_LEN)
loader = DataLoader(ds,
                    batch_size=BATCH_SIZE,
                    collate_fn=collate_fn,
                    num_workers=4,
                    worker_init_fn=worker_init_fn,
                    prefetch_factor=2)

load_iter = iter(loader)

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]


batch = next(load_iter)

print(sp.decode(batch["x_enc"][0].tolist(), out_type=str))
print()
print(sp.decode(batch["x_dec"][0].tolist(), out_type=str))

In [10]:
whereswaldo.train()

for i in range(total_steps):
    optimizer.zero_grad()
    batch, labels = next(load_iter)
    batch = {k : v.to(device) for k, v in batch.items()}
    labels = labels.to(device)

    with autocast(device_type=device.type):
        pred = whereswaldo(**batch)
        loss = F.cross_entropy(pred.view(-1, sp.vocab_size()), labels.view(-1), ignore_index=sp.pad_id())

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(whereswaldo.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    if i % LOG_STEPS == 0:
        print(f"step {i}: loss = {loss.item()} | lr = {scheduler.get_last_lr()[0]}")

step 0: loss = 9.23784351348877 | lr = 9.83284169124877e-08
step 5: loss = 9.201144218444824 | lr = 5.899705014749262e-07
step 10: loss = 9.152044296264648 | lr = 1.0816125860373646e-06
step 15: loss = 9.090925216674805 | lr = 1.5732546705998032e-06
step 20: loss = 8.979242324829102 | lr = 2.0648967551622416e-06
step 25: loss = 8.88906192779541 | lr = 2.55653883972468e-06
step 30: loss = 8.769700050354004 | lr = 3.0481809242871185e-06
step 35: loss = 8.690673828125 | lr = 3.539823008849557e-06


KeyboardInterrupt: 